# Sistema Multi-Agente de Robots de Fútbol con Algoritmo del Lobo Gris

## Coordinación Dinámmica para Búsqueda y Ataque a Portería



In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
try:
    from IPython.display import HTML, display
except ImportError:
    HTML = lambda x: x
    def display(x):
        print(x)
import pandas as pd
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)


## 1. Simulación del Sistema

In [ ]:
# CELDA 1: Simulación con triangulacion, exploracion y percepcion adaptativa
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from dataclasses import dataclass, field

np.random.seed(42)


@dataclass
class Cancha:
    ancho: float = 150
    largo: float = 300
    porteria_a: tuple = (75, 10)
    porteria_b: tuple = (75, 290)
    zona_inicio_a: tuple = (75, 80)
    zona_inicio_b: tuple = (75, 220)
    obstaculos: list = field(default_factory=lambda: [
        {'x': 45, 'y': 145, 'r': 12},
        {'x': 105, 'y': 175, 'r': 12},
        {'x': 75, 'y': 220, 'r': 10}
    ])


class SensorSharp:
    def __init__(self, rango_min=4, rango_max=80):
        self.rango_min = rango_min
        self.rango_max = rango_max

    def medir(self, distancia_real):
        if distancia_real > self.rango_max:
            return None
        distancia_util = max(self.rango_min, distancia_real)
        voltaje = 3.0 - (distancia_util / self.rango_max) * 2.7
        ruido = np.random.normal(0, 0.05)
        return max(0.3, min(3.0, voltaje + ruido))

    def voltaje_a_distancia(self, voltaje):
        if voltaje is None:
            return None
        distancia = (3.0 - voltaje) * self.rango_max / 2.7
        return max(self.rango_min, min(self.rango_max, distancia))


class SensorColor:
    def __init__(self, umbral_naranja=0.50):
        self.umbral = umbral_naranja

    def puntuacion_naranja(self, frec_r, frec_g, frec_b):
        total = frec_r + frec_g + frec_b
        if total <= 0:
            return 0.0
        r_norm = frec_r / total
        g_norm = frec_g / total
        b_norm = frec_b / total
        return (r_norm * 0.8) + (g_norm * 0.4) - (b_norm * 0.3)

    def detectar_balon(self, frec_r, frec_g, frec_b):
        return self.puntuacion_naranja(frec_r, frec_g, frec_b) > self.umbral

    def calibrar_con_muestra(self, frec_r, frec_g, frec_b, alpha=0.08):
        score = self.puntuacion_naranja(frec_r, frec_g, frec_b)
        objetivo = max(0.45, min(0.65, score - 0.02))
        self.umbral = (1 - alpha) * self.umbral + alpha * objetivo


class Encoder:
    def __init__(self, pulsos_por_revolucion=12):
        self.ppr = pulsos_por_revolucion
        self.circunferencia_rueda = 2 * np.pi * 2.3

    def rpm_desde_velocidad(self, velocidad_cm_s):
        if self.circunferencia_rueda <= 0:
            return 0.0
        return (velocidad_cm_s * 60) / self.circunferencia_rueda


class Robot:
    def __init__(self, id_robot, posicion_inicio, porteria_objetivo, nombre="Robot"):
        self.id = id_robot
        self.nombre = nombre
        self.x, self.y = posicion_inicio
        self.porteria = porteria_objetivo
        self.tiene_balon = False
        self.rol = "BUSCADOR"
        self.radio_robot = 5
        self.evasiones_obstaculo = 0

        self.sensor_sharp = SensorSharp()
        self.sensor_color = SensorColor()
        self.encoder = Encoder()

        self.sharp_voltaje = None
        self.sharp_distancia = None
        self.color_detectado = False
        self.color_score = 0.0
        self.frec_r = None
        self.frec_g = None
        self.frec_b = None

        self.angulo_exploracion = np.random.uniform(0, 2 * np.pi)
        self.mapa_detecciones = []

        self.historial_x = [self.x]
        self.historial_y = [self.y]
        self.historial_roles = [self.rol]
        self.historial_sharp_distancia = []
        self.historial_sharp_voltaje = []
        self.historial_color_detectado = []
        self.historial_color_score = []
        self.historial_umbral_color = [self.sensor_color.umbral]
        self.historial_rpm = [0.0]
        self.historial_velocidad = [0.0]

    def distancia_a(self, px, py):
        return np.sqrt((px - self.x)**2 + (py - self.y)**2)

    def distancia_porteria(self):
        return self.distancia_a(self.porteria[0], self.porteria[1])

    def simular_entradas_sensores(self, balon_x, balon_y):
        distancia_real = self.distancia_a(balon_x, balon_y)

        self.sharp_voltaje = self.sensor_sharp.medir(distancia_real)
        self.sharp_distancia = self.sensor_sharp.voltaje_a_distancia(self.sharp_voltaje)

        proximidad = max(0.0, 1.0 - distancia_real / 120.0)
        self.frec_r = max(1.0, 120 + 90 * proximidad + np.random.normal(0, 6))
        self.frec_g = max(1.0, 85 + 35 * proximidad + np.random.normal(0, 6))
        self.frec_b = max(1.0, 110 - 55 * proximidad + np.random.normal(0, 6))

        self.color_score = self.sensor_color.puntuacion_naranja(self.frec_r, self.frec_g, self.frec_b)
        self.color_detectado = self.color_score > self.sensor_color.umbral

        if self.sharp_distancia is not None and self.sharp_distancia <= 25 and self.color_detectado:
            self.sensor_color.calibrar_con_muestra(self.frec_r, self.frec_g, self.frec_b)

        self.historial_sharp_distancia.append(self.sharp_distancia)
        self.historial_sharp_voltaje.append(self.sharp_voltaje)
        self.historial_color_detectado.append(self.color_detectado)
        self.historial_color_score.append(self.color_score)
        self.historial_umbral_color.append(self.sensor_color.umbral)

    def registrar_deteccion(self, tipo_objeto, x, y, confianza, paso_actual):
        self.mapa_detecciones.append({
            'tipo': tipo_objeto,
            'x': float(x),
            'y': float(y),
            'confianza': float(confianza),
            'paso': int(paso_actual)
        })

    def actualizar_mapa_entorno(self, cancha, balon_est_x, balon_est_y, paso_actual):
        radio_vision = 90

        for obs in cancha.obstaculos:
            d = self.distancia_a(obs['x'], obs['y'])
            if d <= radio_vision:
                confianza = max(0.2, 1 - d / radio_vision)
                self.registrar_deteccion('obstaculo', obs['x'], obs['y'], confianza, paso_actual)

        if self.color_detectado and self.sharp_distancia is not None:
            vx = balon_est_x - self.x
            vy = balon_est_y - self.y
            n = np.sqrt(vx**2 + vy**2)
            if n < 1e-6:
                vx = np.cos(self.angulo_exploracion)
                vy = np.sin(self.angulo_exploracion)
                n = 1.0

            bx = self.x + (vx / n) * self.sharp_distancia
            by = self.y + (vy / n) * self.sharp_distancia
            bx = max(5, min(cancha.ancho - 5, bx))
            by = max(5, min(cancha.largo - 5, by))

            confianza = min(1.0, 0.55 + 0.45 * max(0.0, self.color_score - self.sensor_color.umbral + 0.15))
            self.registrar_deteccion('balon_candidato', bx, by, confianza, paso_actual)

    def moverse_hacia(self, px, py, velocidad, dt, obstaculos=None):
        x0, y0 = self.x, self.y
        dx = px - self.x
        dy = py - self.y
        mag = np.sqrt(dx**2 + dy**2)

        if mag < 0.5:
            self.historial_x.append(self.x)
            self.historial_y.append(self.y)
            self.historial_velocidad.append(0.0)
            self.historial_rpm.append(0.0)
            return

        paso = min(velocidad * dt, mag)
        nx = self.x + (dx / mag) * paso
        ny = self.y + (dy / mag) * paso

        for obs in (obstaculos or []):
            ox, oy, r = obs['x'], obs['y'], obs['r']
            dist = np.sqrt((nx - ox)**2 + (ny - oy)**2)
            limite_seguridad = r + self.radio_robot + 2

            if dist < limite_seguridad:
                vx = nx - ox
                vy = ny - oy
                norm = np.sqrt(vx**2 + vy**2)
                if norm < 1e-6:
                    vx, vy, norm = 1.0, 0.0, 1.0

                nx = ox + (vx / norm) * limite_seguridad
                ny = oy + (vy / norm) * limite_seguridad

                tx, ty = -vy / norm, vx / norm
                proyeccion_objetivo = tx * (px - self.x) + ty * (py - self.y)
                sentido = 1 if proyeccion_objetivo >= 0 else -1
                desvio = min(8, paso * 0.25)
                nx += sentido * tx * desvio
                ny += sentido * ty * desvio
                self.evasiones_obstaculo += 1

        self.x = max(5, min(cancha.ancho - 5, nx))
        self.y = max(5, min(cancha.largo - 5, ny))

        distancia_movida = np.sqrt((self.x - x0)**2 + (self.y - y0)**2)
        velocidad_real = distancia_movida / dt if dt > 0 else 0.0
        rpm = self.encoder.rpm_desde_velocidad(velocidad_real)

        self.historial_x.append(self.x)
        self.historial_y.append(self.y)
        self.historial_velocidad.append(velocidad_real)
        self.historial_rpm.append(rpm)


class SimuladorFutbol:
    def __init__(self, cancha, robot_a, robot_b):
        self.cancha = cancha
        self.robot_a = robot_a
        self.robot_b = robot_b
        self.dt = 0.5
        self.umbral_posesion = 20

        self.balon_x = self.cancha.ancho / 2
        self.balon_y = self.cancha.largo / 2

        self.historial_balon = [(self.balon_x, self.balon_y)]
        self.balon_est_x = self.balon_x
        self.balon_est_y = self.balon_y
        self.historial_balon_estimado = [(self.balon_est_x, self.balon_est_y)]

        self.historial_roles_a = []
        self.historial_roles_b = []
        self.gol_anotado = False
        self.robot_goleador = None
        self.paso_gol = None
        self.mapa_global_detecciones = []
        self.ultimo_paso_estimacion = 0
        self.hay_estimacion_util = True

    def _forzar_limites_cancha(self, x, y):
        return max(5, min(self.cancha.ancho - 5, x)), max(5, min(self.cancha.largo - 5, y))

    def actualizar_sensores(self):
        self.robot_a.simular_entradas_sensores(self.balon_x, self.balon_y)
        self.robot_b.simular_entradas_sensores(self.balon_x, self.balon_y)

    def triangular_posicion_balon(self, paso_actual=0):
        x1, y1 = self.robot_a.x, self.robot_a.y
        x2, y2 = self.robot_b.x, self.robot_b.y
        r1 = self.robot_a.sharp_distancia
        r2 = self.robot_b.sharp_distancia

        estimado_actual = np.array([self.balon_est_x, self.balon_est_y], dtype=float)

        if r1 is not None and r2 is not None:
            dx = x2 - x1
            dy = y2 - y1
            d = np.sqrt(dx**2 + dy**2)
            if d > 1e-6 and (abs(r1 - r2) <= d <= (r1 + r2)):
                a = (r1**2 - r2**2 + d**2) / (2 * d)
                h = np.sqrt(max(0.0, r1**2 - a**2))
                x3 = x1 + a * dx / d
                y3 = y1 + a * dy / d
                rx = -dy * (h / d)
                ry = dx * (h / d)
                c1 = np.array([x3 + rx, y3 + ry])
                c2 = np.array([x3 - rx, y3 - ry])
                cand = c1 if np.linalg.norm(c1 - estimado_actual) <= np.linalg.norm(c2 - estimado_actual) else c2
                self.balon_est_x, self.balon_est_y = self._forzar_limites_cancha(float(cand[0]), float(cand[1]))
                self.historial_balon_estimado.append((self.balon_est_x, self.balon_est_y))
                self.ultimo_paso_estimacion = paso_actual
                self.hay_estimacion_util = True
                return

        if r1 is not None and r2 is None:
            vx, vy = self.balon_est_x - x1, self.balon_est_y - y1
            n = np.sqrt(vx**2 + vy**2)
            if n < 1e-6:
                vx, vy, n = 1.0, 0.0, 1.0
            self.balon_est_x, self.balon_est_y = self._forzar_limites_cancha(x1 + (vx / n) * r1, y1 + (vy / n) * r1)
            self.ultimo_paso_estimacion = paso_actual
            self.hay_estimacion_util = True
        elif r2 is not None and r1 is None:
            vx, vy = self.balon_est_x - x2, self.balon_est_y - y2
            n = np.sqrt(vx**2 + vy**2)
            if n < 1e-6:
                vx, vy, n = 1.0, 0.0, 1.0
            self.balon_est_x, self.balon_est_y = self._forzar_limites_cancha(x2 + (vx / n) * r2, y2 + (vy / n) * r2)
            self.ultimo_paso_estimacion = paso_actual
            self.hay_estimacion_util = True

        self.historial_balon_estimado.append((self.balon_est_x, self.balon_est_y))

    def actualizar_percepcion_entorno(self, paso_actual):
        self.robot_a.actualizar_mapa_entorno(self.cancha, self.balon_est_x, self.balon_est_y, paso_actual)
        self.robot_b.actualizar_mapa_entorno(self.cancha, self.balon_est_x, self.balon_est_y, paso_actual)
        ventana = 12
        self.mapa_global_detecciones = [
            d for d in (self.robot_a.mapa_detecciones + self.robot_b.mapa_detecciones)
            if (paso_actual - d['paso']) <= ventana
        ]

    def objetivo_exploracion(self, robot):
        robot.angulo_exploracion += 0.35
        radio = 45 if robot.id == 1 else 55
        cx = self.cancha.ancho / 2 + (-15 if robot.id == 1 else 15)
        cy = self.cancha.largo / 2 + (20 if robot.id == 1 else -20)
        tx = cx + radio * np.cos(robot.angulo_exploracion)
        ty = cy + radio * np.sin(robot.angulo_exploracion * 0.9)
        return self._forzar_limites_cancha(tx, ty)

    def verificar_posesion(self):
        candidatos = []
        for robot in (self.robot_a, self.robot_b):
            if robot.sharp_distancia is not None and robot.sharp_distancia <= self.umbral_posesion and robot.color_detectado:
                candidatos.append((robot.sharp_distancia, robot))

        if not candidatos:
            self.robot_a.tiene_balon = False
            self.robot_b.tiene_balon = False
            return

        candidatos.sort(key=lambda x: x[0])
        poseedor = candidatos[0][1]
        self.robot_a.tiene_balon = (poseedor is self.robot_a)
        self.robot_b.tiene_balon = (poseedor is self.robot_b)

    def asignar_roles_gwo(self):
        d_port_a = self.robot_a.distancia_porteria()
        d_port_b = self.robot_b.distancia_porteria()

        if self.robot_a.tiene_balon:
            self.robot_a.rol = "ATACANTE"
            self.robot_b.rol = "ASISTENTE" if d_port_b <= d_port_a else "BUSCADOR"
        elif self.robot_b.tiene_balon:
            self.robot_b.rol = "ATACANTE"
            self.robot_a.rol = "ASISTENTE" if d_port_a <= d_port_b else "BUSCADOR"
        else:
            self.robot_a.rol = "BUSCADOR"
            self.robot_b.rol = "BUSCADOR"

        self.historial_roles_a.append(self.robot_a.rol)
        self.historial_roles_b.append(self.robot_b.rol)

    def _debe_perseguir_balon(self, robot, paso_actual):
        deteccion_directa = robot.color_detectado and robot.sharp_distancia is not None
        estimacion_reciente = self.hay_estimacion_util and (paso_actual - self.ultimo_paso_estimacion) <= 35
        return deteccion_directa or estimacion_reciente

    def ejecutar_comportamientos(self, paso_actual):
        obstaculos = self.cancha.obstaculos

        if self.robot_a.rol == "ATACANTE":
            self.robot_a.moverse_hacia(self.robot_a.porteria[0], self.robot_a.porteria[1], 200, self.dt, obstaculos)
        elif self.robot_a.rol == "BUSCADOR":
            if self._debe_perseguir_balon(self.robot_a, paso_actual):
                self.robot_a.moverse_hacia(self.balon_est_x, self.balon_est_y, 180, self.dt, obstaculos)
            else:
                tx, ty = self.objetivo_exploracion(self.robot_a)
                self.robot_a.moverse_hacia(tx, ty, 130, self.dt, obstaculos)
        elif self.robot_a.rol == "ASISTENTE":
            self.robot_a.moverse_hacia(self.robot_b.x, self.robot_b.y - 20, 120, self.dt, obstaculos)

        if self.robot_b.rol == "ATACANTE":
            self.robot_b.moverse_hacia(self.robot_b.porteria[0], self.robot_b.porteria[1], 200, self.dt, obstaculos)
        elif self.robot_b.rol == "BUSCADOR":
            if self._debe_perseguir_balon(self.robot_b, paso_actual):
                self.robot_b.moverse_hacia(self.balon_est_x, self.balon_est_y, 180, self.dt, obstaculos)
            else:
                tx, ty = self.objetivo_exploracion(self.robot_b)
                self.robot_b.moverse_hacia(tx, ty, 130, self.dt, obstaculos)
        elif self.robot_b.rol == "ASISTENTE":
            self.robot_b.moverse_hacia(self.robot_a.x, self.robot_a.y + 20, 120, self.dt, obstaculos)

    def actualizar_balon(self):
        if self.robot_a.tiene_balon:
            self.balon_x, self.balon_y = self.robot_a.x, self.robot_a.y
        elif self.robot_b.tiene_balon:
            self.balon_x, self.balon_y = self.robot_b.x, self.robot_b.y

        if self.robot_a.tiene_balon or self.robot_b.tiene_balon:
            self.balon_est_x, self.balon_est_y = self.balon_x, self.balon_y
            self.historial_balon_estimado.append((self.balon_est_x, self.balon_est_y))

        self.historial_balon.append((self.balon_x, self.balon_y))

    def verificar_gol_porteria_b(self):
        umbral_gol = 8
        for robot in (self.robot_a, self.robot_b):
            if robot.tiene_balon and robot.distancia_a(*self.cancha.porteria_b) <= umbral_gol:
                self.gol_anotado = True
                self.robot_goleador = robot.nombre
                self.balon_x, self.balon_y = robot.x, robot.y
                if not self.historial_balon or self.historial_balon[-1] != (self.balon_x, self.balon_y):
                    self.historial_balon.append((self.balon_x, self.balon_y))
                return True
        return False

    def paso(self, paso_actual=None):
        paso = paso_actual if paso_actual is not None else 0
        self.actualizar_sensores()
        self.triangular_posicion_balon(paso)
        self.actualizar_percepcion_entorno(paso)
        self.verificar_posesion()
        self.asignar_roles_gwo()
        self.ejecutar_comportamientos(paso)
        self.actualizar_balon()
        if self.verificar_gol_porteria_b():
            self.paso_gol = paso

    def ejecutar(self, pasos):
        for paso_idx in range(1, pasos + 1):
            self.paso(paso_idx)
            if self.gol_anotado:
                break


# CELDA 2: Ejecutar simulación
cancha = Cancha()
robot_a = Robot(1, cancha.zona_inicio_a, cancha.porteria_b, "Robot A")
robot_b = Robot(2, cancha.zona_inicio_b, cancha.porteria_b, "Robot B")
simulador = SimuladorFutbol(cancha, robot_a, robot_b)

simulador.ejecutar(300)

print(f"Robot A - rol final: {robot_a.rol}, posición: ({robot_a.x:.1f}, {robot_a.y:.1f}), tuvo balón: {robot_a.tiene_balon}")
print(f"Robot B - rol final: {robot_b.rol}, posición: ({robot_b.x:.1f}, {robot_b.y:.1f}), tuvo balón: {robot_b.tiene_balon}")
print(f"Roles A guardados: {len(simulador.historial_roles_a)}")
print(f"Roles B guardados: {len(simulador.historial_roles_b)}")
print(f"Distribución roles A: {Counter(simulador.historial_roles_a)}")
print(f"Distribución roles B: {Counter(simulador.historial_roles_b)}")
print(f"Posición inicial balón (centro): ({cancha.ancho/2:.1f}, {cancha.largo/2:.1f})")
print(f"Evasiones obstáculo - Robot A: {robot_a.evasiones_obstaculo}")
print(f"Evasiones obstáculo - Robot B: {robot_b.evasiones_obstaculo}")


def _promedio_sin_none(vals):
    limpios = [v for v in vals if v is not None]
    return float(np.mean(limpios)) if limpios else None


prom_sharp_a = _promedio_sin_none(robot_a.historial_sharp_distancia)
prom_sharp_b = _promedio_sin_none(robot_b.historial_sharp_distancia)
color_a = sum(robot_a.historial_color_detectado)
color_b = sum(robot_b.historial_color_detectado)
rpm_a = float(np.mean(robot_a.historial_rpm)) if robot_a.historial_rpm else 0.0
rpm_b = float(np.mean(robot_b.historial_rpm)) if robot_b.historial_rpm else 0.0

print(f"Sharp promedio A: {prom_sharp_a:.1f} cm" if prom_sharp_a is not None else "Sharp promedio A: fuera de rango")
print(f"Sharp promedio B: {prom_sharp_b:.1f} cm" if prom_sharp_b is not None else "Sharp promedio B: fuera de rango")
print(f"Detecciones color A: {color_a}")
print(f"Detecciones color B: {color_b}")
print(f"RPM promedio encoder A: {rpm_a:.1f}")
print(f"RPM promedio encoder B: {rpm_b:.1f}")
print(f"Umbral color final A: {robot_a.sensor_color.umbral:.3f}")
print(f"Umbral color final B: {robot_b.sensor_color.umbral:.3f}")
print(f"Detecciones en mapa Robot A: {len(robot_a.mapa_detecciones)}")
print(f"Detecciones en mapa Robot B: {len(robot_b.mapa_detecciones)}")
print(f"Detecciones globales recientes: {len(simulador.mapa_global_detecciones)}")

err_final = np.sqrt((simulador.balon_est_x - simulador.balon_x)**2 + (simulador.balon_est_y - simulador.balon_y)**2)
print(f"Balón estimado final: ({simulador.balon_est_x:.1f}, {simulador.balon_est_y:.1f})")
print(f"Balón real final: ({simulador.balon_x:.1f}, {simulador.balon_y:.1f})")
print(f"Error final triangulación: {err_final:.2f} cm")

if simulador.gol_anotado:
    print(f"Gol en portería B por {simulador.robot_goleador} en el paso {simulador.paso_gol}. Simulación detenida.")
else:
    print("No hubo gol en portería B durante la simulación.")


## 2. Visualización de Resultados

In [ ]:
# CELDA 3: Visualización de trayectorias
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 8))

# Cancha
ax1.set_xlim(0, cancha.ancho)
ax1.set_ylim(0, cancha.largo)
ax1.set_aspect('equal')
ax1.set_title('Trayectorias en la Cancha', fontsize=14, fontweight='bold')
ax1.set_xlabel('X (cm)')
ax1.set_ylabel('Y (cm)')
ax1.plot([0, cancha.ancho, cancha.ancho, 0, 0], [0, 0, cancha.largo, cancha.largo, 0], 'k-', linewidth=2)
ax1.axhline(y=cancha.largo/2, color='gray', linestyle='--', alpha=0.5, label='Línea media')
ax1.plot(cancha.porteria_a[0], cancha.porteria_a[1], 'r^', markersize=14, label='Portería A')
ax1.plot(cancha.porteria_b[0], cancha.porteria_b[1], 'bs', markersize=14, label='Portería B')

# Obstáculos
for idx, obs in enumerate(cancha.obstaculos):
    circulo = plt.Circle((obs['x'], obs['y']), obs['r'], color='dimgray', alpha=0.35,
                         label='Obstáculo' if idx == 0 else None)
    ax1.add_patch(circulo)
    ax1.plot(obs['x'], obs['y'], 'kx', markersize=6)

ax1.plot(robot_a.historial_x, robot_a.historial_y, 'r-', alpha=0.7, linewidth=2, label='Robot A')
ax1.plot(robot_b.historial_x, robot_b.historial_y, 'b-', alpha=0.7, linewidth=2, label='Robot B')
ax1.plot(robot_a.historial_x[0], robot_a.historial_y[0], 'ro', markersize=8)
ax1.plot(robot_b.historial_x[0], robot_b.historial_y[0], 'bo', markersize=8)
ax1.plot(robot_a.historial_x[-1], robot_a.historial_y[-1], 'r*', markersize=12)
ax1.plot(robot_b.historial_x[-1], robot_b.historial_y[-1], 'b*', markersize=12)
balon_xs = [b[0] for b in simulador.historial_balon]
balon_ys = [b[1] for b in simulador.historial_balon]
balon_est_xs = [b[0] for b in simulador.historial_balon_estimado]
balon_est_ys = [b[1] for b in simulador.historial_balon_estimado]
ax1.plot(balon_xs, balon_ys, color='orange', alpha=0.85, linewidth=2.5, linestyle='--', label='Traza balón real')
ax1.plot(balon_est_xs, balon_est_ys, color='purple', alpha=0.75, linewidth=2.0, linestyle=':', label='Traza balón estimado')
ax1.scatter(balon_xs[0], balon_ys[0], color='gold', edgecolors='black', s=90, marker='o', zorder=5, label='Inicio balón')
ax1.scatter(balon_xs[-1], balon_ys[-1], color='darkorange', edgecolors='black', s=110, marker='*', zorder=6, label='Fin balón')
ax1.legend(loc='best', fontsize=9)
ax1.grid(True, alpha=0.3)

# Distancia a portería
num_pasos = len(simulador.historial_roles_a)
tiempo_sim = np.arange(num_pasos) * 0.5

dist_a = [np.sqrt((robot_a.historial_x[i] - robot_a.porteria[0])**2 +
                  (robot_a.historial_y[i] - robot_a.porteria[1])**2)
          for i in range(min(num_pasos, len(robot_a.historial_x)))]

dist_b = [np.sqrt((robot_b.historial_x[i] - robot_b.porteria[0])**2 +
                  (robot_b.historial_y[i] - robot_b.porteria[1])**2)
          for i in range(min(num_pasos, len(robot_b.historial_x)))]

ax2.plot(tiempo_sim[:len(dist_a)], dist_a, 'r-', linewidth=2, label='Robot A')
ax2.plot(tiempo_sim[:len(dist_b)], dist_b, 'b-', linewidth=2, label='Robot B')
ax2.set_xlabel('Tiempo (s)')
ax2.set_ylabel('Distancia a Portería (cm)')
ax2.set_title('Distancia a Portería vs Tiempo', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# Visualizaciones separadas: Robot A, Robot B y Balón
fig_sep, axes = plt.subplots(1, 3, figsize=(20, 6))

def preparar_cancha(ax, titulo):
    ax.set_xlim(0, cancha.ancho)
    ax.set_ylim(0, cancha.largo)
    ax.set_aspect('equal')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xlabel('X (cm)')
    ax.set_ylabel('Y (cm)')
    ax.plot([0, cancha.ancho, cancha.ancho, 0, 0], [0, 0, cancha.largo, cancha.largo, 0], 'k-', linewidth=1.5)
    ax.axhline(y=cancha.largo/2, color='gray', linestyle='--', alpha=0.4)
    ax.plot(cancha.porteria_a[0], cancha.porteria_a[1], 'r^', markersize=8)
    ax.plot(cancha.porteria_b[0], cancha.porteria_b[1], 'bs', markersize=8)
    for obs in cancha.obstaculos:
        ax.add_patch(plt.Circle((obs['x'], obs['y']), obs['r'], color='dimgray', alpha=0.25))
    ax.grid(True, alpha=0.25)

# Robot A
preparar_cancha(axes[0], 'Trayectoria Robot A')
axes[0].plot(robot_a.historial_x, robot_a.historial_y, color='red', linewidth=2, label='Ruta A')
axes[0].scatter(robot_a.historial_x[0], robot_a.historial_y[0], color='red', s=45, label='Inicio A')
axes[0].scatter(robot_a.historial_x[-1], robot_a.historial_y[-1], color='darkred', marker='*', s=90, label='Fin A')
axes[0].legend(loc='best', fontsize=8)

# Robot B
preparar_cancha(axes[1], 'Trayectoria Robot B')
axes[1].plot(robot_b.historial_x, robot_b.historial_y, color='blue', linewidth=2, label='Ruta B')
axes[1].scatter(robot_b.historial_x[0], robot_b.historial_y[0], color='blue', s=45, label='Inicio B')
axes[1].scatter(robot_b.historial_x[-1], robot_b.historial_y[-1], color='navy', marker='*', s=90, label='Fin B')
axes[1].legend(loc='best', fontsize=8)

# Balón
preparar_cancha(axes[2], 'Trayectoria del Balón')
axes[2].plot(balon_xs, balon_ys, color='orange', linewidth=2.5, linestyle='--', label='Ruta balón real')
axes[2].plot(balon_est_xs, balon_est_ys, color='purple', linewidth=2.0, linestyle=':', label='Ruta balón estimado')
axes[2].scatter(balon_xs[0], balon_ys[0], color='gold', edgecolors='black', s=70, label='Inicio balón')
axes[2].scatter(balon_xs[-1], balon_ys[-1], color='darkorange', edgecolors='black', marker='*', s=110, label='Fin balón')
axes[2].legend(loc='best', fontsize=8)

plt.tight_layout()
plt.show()


## 3. Análisis de Roles Asignados

In [ ]:
# CELDA 4: Análisis de roles
num_pasos = len(simulador.historial_roles_a)
tiempo_sim = np.arange(num_pasos) * 0.5

mapa_roles = {'BUSCADOR': 1, 'ATACANTE': 2, 'ASISTENTE': 3}
roles_a = [mapa_roles.get(r, 0) for r in simulador.historial_roles_a]
roles_b = [mapa_roles.get(r, 0) for r in simulador.historial_roles_b]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6))

ax1.step(tiempo_sim, roles_a, where='mid', color='red', linewidth=2)
ax1.set_title('Evolución de Roles - Robot A', fontsize=14, fontweight='bold')
ax1.set_ylabel('Rol')
ax1.set_yticks([1, 2, 3])
ax1.set_yticklabels(['BUSCADOR', 'ATACANTE', 'ASISTENTE'])
ax1.grid(True, alpha=0.3)

ax2.step(tiempo_sim, roles_b, where='mid', color='blue', linewidth=2)
ax2.set_title('Evolución de Roles - Robot B', fontsize=14, fontweight='bold')
ax2.set_xlabel('Tiempo (s)')
ax2.set_ylabel('Rol')
ax2.set_yticks([1, 2, 3])
ax2.set_yticklabels(['BUSCADOR', 'ATACANTE', 'ASISTENTE'])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nRoles A: {Counter(simulador.historial_roles_a)}")
print(f"Roles B: {Counter(simulador.historial_roles_b)}")

## 4. Métricas de Desempeño

In [ ]:
# Calcular métricas
distancia_final_a = np.sqrt((robot_a.x - robot_a.porteria[0])**2 + (robot_a.y - robot_a.porteria[1])**2)
distancia_final_b = np.sqrt((robot_b.x - robot_b.porteria[0])**2 + (robot_b.y - robot_b.porteria[1])**2)

distancia_inicial_a = np.sqrt((cancha.zona_inicio_a[0] - robot_a.porteria[0])**2 + (cancha.zona_inicio_a[1] - robot_a.porteria[1])**2)
distancia_inicial_b = np.sqrt((cancha.zona_inicio_b[0] - robot_b.porteria[0])**2 + (cancha.zona_inicio_b[1] - robot_b.porteria[1])**2)

progreso_a = ((distancia_inicial_a - distancia_final_a) / distancia_inicial_a) * 100
progreso_b = ((distancia_inicial_b - distancia_final_b) / distancia_inicial_b) * 100

print("\n=== MÉTRICAS DE DESEMPEÑO ===")
print(f"\nRobot A:")
print(f"  Posición inicial: {cancha.zona_inicio_a}")
print(f"  Posición final: ({robot_a.x:.1f}, {robot_a.y:.1f})")
print(f"  Portería objetivo: {robot_a.porteria}")
print(f"  Distancia inicial: {distancia_inicial_a:.1f} cm")
print(f"  Distancia final: {distancia_final_a:.1f} cm")
print(f"  Progreso: {progreso_a:.1f}%")
print(f"  Posesiones balón: {robot_a.tiene_balon}")

print(f"\nRobot B:")
print(f"  Posición inicial: {cancha.zona_inicio_b}")
print(f"  Posición final: ({robot_b.x:.1f}, {robot_b.y:.1f})")
print(f"  Portería objetivo: {robot_b.porteria}")
print(f"  Distancia inicial: {distancia_inicial_b:.1f} cm")
print(f"  Distancia final: {distancia_final_b:.1f} cm")
print(f"  Progreso: {progreso_b:.1f}%")
print(f"  Posesiones balón: {robot_b.tiene_balon}")

# Distancia recorrida
def calcular_distancia_recorrida(hist_x, hist_y):
    dist = 0
    for i in range(1, len(hist_x)):
        dist += np.sqrt((hist_x[i] - hist_x[i-1])**2 + (hist_y[i] - hist_y[i-1])**2)
    return dist

dist_recorrida_a = calcular_distancia_recorrida(robot_a.historial_x, robot_a.historial_y)
dist_recorrida_b = calcular_distancia_recorrida(robot_b.historial_x, robot_b.historial_y)

print(f"\nDistancia recorrida - Robot A: {dist_recorrida_a:.1f} cm")
print(f"Distancia recorrida - Robot B: {dist_recorrida_b:.1f} cm")

# Gráficas de rendimiento
def contar_cambios_roles(historial_roles):
    return sum(1 for i in range(1, len(historial_roles)) if historial_roles[i] != historial_roles[i-1])

cambios_roles_a = contar_cambios_roles(simulador.historial_roles_a)
cambios_roles_b = contar_cambios_roles(simulador.historial_roles_b)

fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Progreso porcentual hacia portería
axs[0].bar(['Robot A', 'Robot B'], [progreso_a, progreso_b], color=['red', 'blue'], alpha=0.75)
axs[0].set_title('Progreso a Portería (%)', fontweight='bold')
axs[0].set_ylabel('Porcentaje (%)')
axs[0].grid(axis='y', alpha=0.3)

# Distancia total recorrida
axs[1].bar(['Robot A', 'Robot B'], [dist_recorrida_a, dist_recorrida_b], color=['red', 'blue'], alpha=0.75)
axs[1].set_title('Distancia Recorrida (cm)', fontweight='bold')
axs[1].set_ylabel('Centímetros')
axs[1].grid(axis='y', alpha=0.3)

# Dinamismo de roles
axs[2].bar(['Robot A', 'Robot B'], [cambios_roles_a, cambios_roles_b], color=['red', 'blue'], alpha=0.75)
axs[2].set_title('Cambios de Rol', fontweight='bold')
axs[2].set_ylabel('Cantidad')
axs[2].grid(axis='y', alpha=0.3)

plt.suptitle('Resumen de Rendimiento del Sistema Multi-Agente', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Cambios de rol - Robot A: {cambios_roles_a}")
print(f"Cambios de rol - Robot B: {cambios_roles_b}")


## 5. Datos para Regresión Múltiple del Sensor de Color

Generación de dataset sintético (RGB + objetivo continuo) para entrenar regresión múltiple.

In [ ]:
# Dataset sintético para regresión múltiple del sensor de color
# Variables predictoras (X): frec_r, frec_g, frec_b, distancia_real_cm, balon_presente
# Variable objetivo (y): score_naranja_objetivo (continua)

n_muestras = 1200

# Distancia simulada al balón (afecta intensidad de señal)
distancia_real_cm = np.random.uniform(4, 120, n_muestras)

# 1 = balón naranja presente, 0 = fondo/no balón
balon_presente = np.random.binomial(1, 0.65, n_muestras)

proximidad = np.clip(1 - distancia_real_cm / 120, 0, 1)

# Señales RGB simuladas (frecuencias) con ruido
frec_r = np.where(
    balon_presente == 1,
    120 + 100 * proximidad + np.random.normal(0, 8, n_muestras),
    90 + 20 * np.random.rand(n_muestras) + np.random.normal(0, 10, n_muestras)
)

frec_g = np.where(
    balon_presente == 1,
    85 + 45 * proximidad + np.random.normal(0, 7, n_muestras),
    85 + 25 * np.random.rand(n_muestras) + np.random.normal(0, 9, n_muestras)
)

frec_b = np.where(
    balon_presente == 1,
    115 - 55 * proximidad + np.random.normal(0, 7, n_muestras),
    95 + 30 * np.random.rand(n_muestras) + np.random.normal(0, 9, n_muestras)
)

# Evitar valores no fisicos
frec_r = np.clip(frec_r, 1, None)
frec_g = np.clip(frec_g, 1, None)
frec_b = np.clip(frec_b, 1, None)

# Normalización RGB y objetivo continuo para regresión
total = frec_r + frec_g + frec_b
r_norm = frec_r / total
g_norm = frec_g / total
b_norm = frec_b / total

score_naranja_objetivo = 0.8 * r_norm + 0.4 * g_norm - 0.3 * b_norm + np.random.normal(0, 0.015, n_muestras)
score_naranja_objetivo = np.clip(score_naranja_objetivo, 0, 1)

# Etiqueta opcional (por si luego quieres comparar con clasificación)
etiqueta_naranja = (score_naranja_objetivo > 0.50).astype(int)

# DataFrame final
regresion_color_df = pd.DataFrame({
    'frec_r': frec_r,
    'frec_g': frec_g,
    'frec_b': frec_b,
    'distancia_real_cm': distancia_real_cm,
    'balon_presente': balon_presente,
    'score_naranja_objetivo': score_naranja_objetivo,
    'etiqueta_naranja': etiqueta_naranja
})

# Guardar dataset
from pathlib import Path
ruta_csv = Path.cwd() / 'datos_regresion_sensor_color.csv'
regresion_color_df.to_csv(ruta_csv, index=False)

print("Dataset generado para regresion multiple del sensor de color")
print(f"Muestras: {len(regresion_color_df)}")
print(f"Archivo: {ruta_csv}")
print()
print("Columnas:", list(regresion_color_df.columns))
print()
print("Primeras filas:")
display(regresion_color_df.head(10))
print()
print("Correlación con score_naranja_objetivo:")
print(regresion_color_df[['frec_r','frec_g','frec_b','score_naranja_objetivo']].corr()['score_naranja_objetivo'])

# Entrenamiento de regresión lineal múltiple (sin dependencias externas)
features = ['frec_r', 'frec_g', 'frec_b', 'distancia_real_cm', 'balon_presente']
X = regresion_color_df[features].to_numpy(dtype=float)
y = regresion_color_df['score_naranja_objetivo'].to_numpy(dtype=float)

rng = np.random.default_rng(42)
indices = rng.permutation(len(X))
n_train = int(0.8 * len(X))
idx_train = indices[:n_train]
idx_test = indices[n_train:]

X_train = X[idx_train]
X_test = X[idx_test]
y_train = y[idx_train]
y_test = y[idx_test]

X_train_aug = np.column_stack([np.ones(len(X_train)), X_train])
coef = np.linalg.lstsq(X_train_aug, y_train, rcond=None)[0]
intercepto = float(coef[0])
pesos = coef[1:]

def predecir_regresion_multiple(X_in):
    return intercepto + X_in @ pesos

y_pred = np.clip(predecir_regresion_multiple(X_test), 0.0, 1.0)
mse = float(np.mean((y_test - y_pred) ** 2))
rmse = float(np.sqrt(mse))
mae = float(np.mean(np.abs(y_test - y_pred)))
ss_res = float(np.sum((y_test - y_pred) ** 2))
ss_tot = float(np.sum((y_test - np.mean(y_test)) ** 2))
r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else 0.0

coeficientes_df = pd.DataFrame({
    'feature': features,
    'coeficiente': pesos
})

print("\\n=== MODELO DE REGRESIÓN MÚLTIPLE ENTRENADO ===")
print(f"Intercepto: {intercepto:.6f}")
display(coeficientes_df)
print(f"R2 (test): {r2:.4f}")
print(f"MAE (test): {mae:.4f}")
print(f"RMSE (test): {rmse:.4f}")

predicciones_df = pd.DataFrame({
    'y_real': y_test[:10],
    'y_pred': y_pred[:10],
    'error_abs': np.abs(y_test[:10] - y_pred[:10])
})
print("\\nPrimeras 10 predicciones (test):")
display(predicciones_df)


Dataset generado para regresión múltiple del sensor de color
Muestras: 1200
Archivo: c:\Users\wsteb\OneDrive\Documentos\Proyectos\2026-01\RobotUsa\datos_regresion_sensor_color.csv

Columnas: ['frec_r', 'frec_g', 'frec_b', 'distancia_real_cm', 'balon_presente', 'score_naranja_objetivo', 'etiqueta_naranja']

Primeras filas:


,frec_r,frec_g,frec_b,distancia_real_cm,balon_presente,score_naranja_objetivo,etiqueta_naranja
0,103.839256,92.241121,113.755824,47.446654,0,0.313651,0
1,123.090509,90.673943,115.220413,114.282860,1,0.309019,0
2,132.439747,98.695017,100.865631,88.911297,1,0.337730,0
3,152.349386,98.909295,86.753410,73.444384,1,0.388142,0
4,209.303678,118.115885,54.337785,22.098162,1,0.512294,1
5,106.012406,89.128490,126.251923,22.095364,0,0.289621,0
6,120.325400,100.978090,110.240543,10.737699,0,0.306426,0
7,128.196592,90.915490,103.557806,104.476433,1,0.347865,0
8,158.347636,100.777526,82.458403,73.729341,1,0.410809,0
9,102.859282,84.142421,76.710443,86.136419,0,0.356314,0



Correlación con score_naranja_objetivo:
frec_r                    0.956708
frec_g                    0.657042
frec_b                   -0.912318
score_naranja_objetivo    1.000000
Name: score_naranja_objetivo, dtype: float64
\n=== MODELO DE REGRESIÓN MÚLTIPLE ENTRENADO ===
Intercepto: 0.350212


,feature,coeficiente
0,frec_r,0.001157
1,frec_g,0.000136
2,frec_b,-0.001836
3,distancia_real_cm,0.000044
4,balon_presente,0.010480


R2 (test): 0.9664
MAE (test): 0.0133
RMSE (test): 0.0161
\nPrimeras 10 predicciones (test):


,y_real,y_pred,error_abs
0,0.416770,0.410708,0.006062
1,0.522806,0.523775,0.000969
2,0.391352,0.396174,0.004821
3,0.258962,0.258556,0.000406
4,0.528094,0.507321,0.020774
5,0.274030,0.269989,0.004040
6,0.250117,0.268453,0.018336
7,0.232339,0.249648,0.017308
8,0.375944,0.367984,0.007960
9,0.463333,0.491830,0.028497
